In [ ]:
import os  # reads settings from the environment

import pandas as pd  # pandas, always nicknamed "pd"
from dotenv import load_dotenv  # loads .env, same as loader.py
from sqlalchemy import URL, create_engine  # the connection pandas prefers

pd.set_option("display.max_columns", None)  # show all columns, never "..."

load_dotenv()

url = URL.create(
    "postgresql+psycopg",                   # Postgres, through psycopg
    username=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    host=os.getenv("DB_HOST"),
    port=int(os.getenv("DB_PORT")),         # must be a number here, not text
    database=os.getenv("DB_NAME"),
)
engine = create_engine(url)                 # a reusable door into the database

In [ ]:
query = "SELECT * FROM pd_features_build WHERE sample_set = 'train'"
df = pd.read_sql(query, engine)   # run the query, keep the result as a table
df.shape                          # (rows, columns)

In [ ]:
print(df.head(10).T.to_string())

In [ ]:
df.info()

In [ ]:
print(df.describe().T.to_string())

In [ ]:
for col in ["amortization_type", "prepayment_penalty_indicator",
            "interest_only_indicator", "property_type", "channel"]:
    print(df[col].value_counts(dropna=False))
    print()

In [ ]:
print(df["msa"].isna().mean())
print(df["msa"].nunique())
print(df["msa"].value_counts().head(10))

In [ ]:
print(df["postal_code"].nunique())
print(df["property_state"].nunique())

In [ ]:
print("--- check 3: LTV over 100 ---")
high_ltv = df[df["original_ltv"] > 100]
print(len(high_ltv))
print(high_ltv["harp_indicator"].value_counts())

print("--- check 4: rate spread range, and the odd 276/288 terms ---")
print(df["rate_spread"].describe())
print(df[df["loan_term"].isin([276, 288])]["harp_indicator"].value_counts())

print("--- check 5: credit scores at the very bottom ---")
print((df["credit_score"] == 300).sum())
print((df["credit_score"] < 500).sum())

In [ ]:
missing = df.isna().sum()
missing = missing[missing > 0]
print(pd.DataFrame({"missing": missing, "pct": (missing / len(df) * 100).round(3)}))

In [ ]:
fill_numbers = ["credit_score", "number_of_borrowers", "number_of_units",
                "original_cltv", "original_ltv", "mi_percentage"]
print(df[fill_numbers].median())
print(df["first_time_homebuyer"].mode())
print(df["property_type"].mode())

print(df.groupby(df["original_dti_ratio"].isna())["target"].agg(["count", "mean"]))


In [ ]:
df["year"] = "20" + df["loan_sequence_number"].str[1:3]
print(df.groupby("year")["target"].agg(["count", "mean"]))

In [ ]:
num_cols = ["credit_score", "original_ltv", "original_cltv", "original_dti_ratio",
            "original_upb", "rate_spread", "loan_term", "mi_percentage",
            "number_of_borrowers", "number_of_units"]
print(df.groupby("year")[num_cols].mean().round(2).T)

In [ ]:
for col in ["loan_purpose", "channel", "occupancy_status", "property_type",
            "first_time_homebuyer", "harp_indicator"]:
    print(pd.crosstab(df["year"], df[col], normalize="index").round(3))
    print()

In [ ]:
for col in ["property_type", "first_time_homebuyer"]:
    print(pd.crosstab(df["year"], df[col], normalize="index").round(3))
    print()

y2012 = df[df["year"] == "2012"]
print(y2012.groupby("harp_indicator")[["target", "rate_spread"]].mean())

In [ ]:
print((df["original_ltv"] > 100).sum())

In [ ]:
print(df.groupby(df["original_dti_ratio"].isna())["target"].agg(["count", "mean"]))

In [ ]:
# The model's features: D-057 candidates plus rate_spread,
# minus the 5 fields of D-060 and harp_indicator (D-062)
features = ["credit_score", "original_ltv", "original_cltv", "original_dti_ratio",
            "original_upb", "rate_spread", "loan_term", "mi_percentage",
            "number_of_borrowers", "number_of_units", "first_time_homebuyer",
            "occupancy_status", "channel", "property_state", "property_type",
            "loan_purpose"]
train = df[["target"] + features].copy()

# D-062: channel as retail (R) vs third party (TP)
train["channel"] = train["channel"].replace({"B": "TP", "C": "TP", "T": "TP"})

# D-061: fill small gaps with values learned on train
fill_values = train[["credit_score", "original_ltv", "original_cltv",
                     "original_dti_ratio", "mi_percentage",
                     "number_of_borrowers", "number_of_units"]].median().to_dict()
fill_values["first_time_homebuyer"] = train["first_time_homebuyer"].mode()[0]
fill_values["property_type"] = train["property_type"].mode()[0]
train = train.fillna(fill_values)

print(train.shape)
print(fill_values)
print(train.isna().sum().sum())
print(train["channel"].value_counts())

In [ ]:
from optbinning import OptimalBinning

optb = OptimalBinning(name="credit_score", dtype="numerical",
                      monotonic_trend="auto_asc_desc",
                      min_bin_size=0.05, max_n_bins=8)
optb.fit(train["credit_score"], train["target"])

table = optb.binning_table.build()
print(table[["Bin", "Count", "Count (%)", "Non-event", "Event",
             "Event rate", "WoE", "IV"]].to_string())

In [ ]:
binners = {"credit_score": optb}

for col in ["original_ltv", "original_dti_ratio", "rate_spread", "original_upb"]:
    ob = OptimalBinning(name=col, dtype="numerical",
                        monotonic_trend="auto_asc_desc",
                        min_bin_size=0.05, max_n_bins=8)
    ob.fit(train[col], train["target"])
    binners[col] = ob
    t = ob.binning_table.build()
    print(col, "IV =", round(t.loc["Totals", "IV"], 3))
    print(t[["Bin", "Count (%)", "Event rate", "WoE"]].iloc[:-3].to_string())
    print()